# 03 — Hybrid search and answers

Retrieval and generation go through `src.opensearch` and `src.chat`.

The search pipeline `nlp-search-pipeline` combines BM25 and kNN with min-max normalization and an arithmetic mean. Weights are 0.3 for keyword search and 0.7 for vectors. The app creates that pipeline when it creates the index.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.chat import citations_from_hits, prepare_prompt
from src.constants import OLLAMA_MODEL_NAME
from src.opensearch import SEARCH_PIPELINE_BODY, SEARCH_PIPELINE_NAME, hybrid_search

print(SEARCH_PIPELINE_NAME)
print(SEARCH_PIPELINE_BODY["phase_results_processors"][0]["normalization-processor"]["combination"])
print("Ollama model:", OLLAMA_MODEL_NAME)


`prepare_prompt` embeds the question, runs hybrid search, and builds the same prompt the chatbot sends to Ollama. Leave the question empty to skip the live call.


In [ ]:
QUESTION = ""  # for example: "What does this document say about sea level?"

if not QUESTION:
    print("Set QUESTION to run hybrid search against the documents index.")
else:
    prompt, citations = prepare_prompt(
        QUESTION,
        use_hybrid_search=True,
        num_results=5,
        chat_history=[],
    )
    for cite in citations:
        print(cite["document_name"], cite["score"])
        print(cite["snippet"])
        print("---")
    print(prompt[:500])
